# Chapter 16 &mdash; SAT in Practice: Solvers, DIMACS, and Equisatisfiability

**Concept 12 of the Chapter 16 decomposition:** *SAT in Practice: Solvers, DIMACS, and Equisatisfiability*

NP-completeness is a worst-case statement; real solvers handle millions of variables.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-SAT-In-Practice/Concept-SAT-In-Practice.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


NP-completeness is a **worst-case** statement, and practice diverges from it sharply.
Modern CDCL solvers routinely dispatch industrial instances with **millions** of
variables &mdash; hardware verification, planning, scheduling, bounded model checking.

Two things you need to use one:

* **DIMACS CNF** &mdash; the universal input format: a `p cnf <vars> <clauses>` header,
  then one clause per line as space-separated integers terminated by `0`.
* **Tseitin encoding** &mdash; converting an arbitrary circuit or formula to CNF by
  naming each gate with a fresh variable. The result is **equisatisfiable**, not
  equivalent: it has extra variables, but it is satisfiable exactly when the original
  is, and it is **linear** in size rather than exponential.

The practical takeaway: "NP-complete" means *do not expect a polynomial guarantee*, not
*give up*.

## 2. Definitions

### DIMACS

In [ ]:
# --- a tiny CNF toolkit -------------------------------------------------
# A literal is an int: 3 means x3, -3 means NOT x3.
# A clause is a tuple of literals; a formula is a list of clauses.
from itertools import product

def nvars(F):
    return max((abs(l) for c in F for l in c), default=0)

def evaluate(F, assign):
    # assign: dict var -> bool
    return all(any(assign[abs(l)] == (l > 0) for l in c) for c in F)

def brute_sat(F):
    n = nvars(F)
    for bits in product([False, True], repeat=n):
        a = {i + 1: bits[i] for i in range(n)}
        if evaluate(F, a): return a
    return None

def show_cnf(F):
    def lit(l): return ("x%d" % l) if l > 0 else ("~x%d" % -l)
    return " AND ".join("(" + " OR ".join(lit(l) for l in c) + ")" for c in F)


def to_dimacs(F, comment=None):
    lines = []
    if comment: lines.append("c " + comment)
    lines.append("p cnf %d %d" % (nvars(F), len(F)))
    for c_ in F:
        lines.append(" ".join(str(l) for l in c_) + " 0")
    return "\n".join(lines)

def from_dimacs(text):
    F = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line[0] in 'cp': continue
        lits = [int(t) for t in line.split() if t != '0']
        if lits: F.append(tuple(lits))
    return F

### Tseitin encoding: name every gate

In [ ]:
class Tseitin:
    def __init__(self, nin):
        self.n = nin
        self.F = []
    def fresh(self):
        self.n += 1; return self.n
    def AND(self, a, b):
        g = self.fresh()
        self.F += [(-g, a), (-g, b), (g, -a, -b)]
        return g
    def OR(self, a, b):
        g = self.fresh()
        self.F += [(-g, a, b), (g, -a), (g, -b)]
        return g
    def NOT(self, a):
        return -a
    def assert_true(self, a):
        self.F.append((a,))

## 3. Tests

A formula in DIMACS.

In [ ]:
F = [(1, -2, 3), (-1, 2), (2, 3)]
print(to_dimacs(F, comment="a tiny example"))
assert from_dimacs(to_dimacs(F)) == F
print("\nround-trips exactly")

**Tseitin:** encode $(x_1 \wedge x_2)\vee x_3$ without blowing up.

In [ ]:
T = Tseitin(3)
g1 = T.AND(1, 2)
g2 = T.OR(g1, 3)
T.assert_true(g2)
print("clauses :", show_cnf(T.F))
print("variables: 3 original + %d gate = %d" % (T.n - 3, T.n))

**Equisatisfiable, not equivalent.** The gate variables are extra.

In [ ]:
def original(a):
    return (a[1] and a[2]) or a[3]
sols = []
for bits in product([False, True], repeat=T.n):
    a = {i + 1: bits[i] for i in range(T.n)}
    if evaluate(T.F, a): sols.append(a)
print("encoded formula has %d satisfying assignments" % len(sols))
orig = [dict(zip((1,2,3), b)) for b in product([False, True], repeat=3)
        if original(dict(zip((1,2,3), b)))]
print("original formula has %d over its 3 variables" % len(orig))
proj = {tuple(sorted((k, v) for k, v in s.items() if k <= 3)) for s in sols}
want = {tuple(sorted(o.items())) for o in orig}
print("projections match? ", proj == want)
assert proj == want

And it is **linear**, where naive CNF conversion is exponential.

In [ ]:
def naive_blowup(k):
    # (a1 AND b1) OR (a2 AND b2) OR ... distributes into 2^k clauses
    return 2 ** k
def tseitin_size(k):
    return 3 * k + 3 * (k - 1) + 1
print("%-6s %-18s %s" % ("k", "naive CNF clauses", "Tseitin clauses"))
for k in [2, 4, 8, 16, 24]:
    print("%-6d %-18s %d" % (k, format(naive_blowup(k), ','), tseitin_size(k)))
assert tseitin_size(24) < naive_blowup(24)

A DPLL solver, to show what the real ones start from.

In [ ]:
def dpll(F, assign=None):
    assign = dict(assign or {})
    F2 = []
    for c_ in F:
        if any(assign.get(abs(l)) == (l > 0) for l in c_): continue
        nc = tuple(l for l in c_ if abs(l) not in assign)
        if not nc: return None
        F2.append(nc)
    if not F2: return assign
    unit = next((c_[0] for c_ in F2 if len(c_) == 1), None)
    if unit is not None:
        assign[abs(unit)] = unit > 0
        return dpll(F, assign)
    l = F2[0][0]
    for val in (l > 0, l <= 0):
        a2 = dict(assign); a2[abs(l)] = val
        r = dpll(F, a2)
        if r is not None: return r
    return None

for F_ in [[(1, -2, 3), (-1, 2), (2, 3)], [(1,), (-1,)]]:
    r = dpll(F_)
    print("  %-30s -> %s" % (show_cnf(F_)[:30], "SAT" if r else "UNSAT"))
    assert (r is not None) == (brute_sat(F_) is not None)

What real solvers add on top.

In [ ]:
FEATURES = ["conflict-driven clause learning (CDCL)",
            "two-watched-literal unit propagation",
            "VSIDS activity-based branching",
            "restarts and clause-database reduction",
            "preprocessing and in-processing"]
for f in FEATURES: print("  *", f)
print()
print("NP-completeness is worst case.  Industrial instances are structured,")
print("and structure is what these techniques exploit.")

## 4. Exercises


1. Write the Tseitin clauses for XOR. How many?
2. Download MiniSat or CaDiCaL and feed it a DIMACS file from this notebook.
3. Why is "equisatisfiable" enough? When would you need "equivalent"?

In [ ]:
# Your work for the exercises above.